# Himawari-9 ingest + processing (Kaggle, CPU)

Bronze → silver for one chunk of days (set `START`/`END` below) over Ho Chi Minh City, using
[hcmc-storm-nowcasting](https://github.com/ngocquangbiii06/hcmc-storm-nowcasting).

**Settings:** Internet **on**, accelerator **None**. Run with **Save Version → Save & Run All** so it runs in
the background (the PC can be turned off). Run the month in chunks (e.g. one week per version or per notebook copy):
a failed Kaggle run saves nothing, so short runs lose less. Output (`/kaggle/working/data`): daily silver NetCDF,
Parquet catalog, a gold Zarr for the chunk, `timings.csv` (with disk/RAM per day) and `run.log`.

In [ ]:
!git clone -q --depth 1 https://github.com/ngocquangbiii06/hcmc-storm-nowcasting.git /tmp/stormcast
!pip install -q -e "/tmp/stormcast"
import os, sys, importlib
os.chdir('/tmp/stormcast'); sys.path.insert(0, '/tmp/stormcast/src')
# fail loudly here rather than 141 times per day later
for mod in ['satpy', 'pyresample', 'xarray', 'zarr', 'boto3'] + []:
    importlib.import_module(mod)
print(sys.version.split()[0], 'dependencies ok')

In [ ]:
import time, logging, shutil
from dataclasses import replace
import pandas as pd
from stormcast import pipeline
from stormcast.config import load_config
from stormcast.ingest.himawari import wanted_segments

START, END = '2025-08-01', '2025-08-02'  # one chunk per run
TIME_BUDGET_H = 11.0  # Kaggle stops a run at 12 h; leave time for build + report

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True,
                    handlers=[logging.StreamHandler(), logging.FileHandler('/kaggle/working/run.log')])
logging.getLogger('satpy').setLevel(logging.WARNING)  # its per-file INFO lines flood the Kaggle log viewer

cfg = load_config('configs/default.yaml')
cfg = replace(
    cfg,
    period=replace(cfg.period, start=START, end=END),
    paths=replace(cfg.paths, data_root='/kaggle/working/data', reports='/kaggle/working/reports'),
    himawari=replace(cfg.himawari, keep_raw=False, download_workers=32),
    processing=replace(cfg.processing, workers=os.cpu_count()),
)

def resources():
    """Free disk (GB) where things can fill up, plus available RAM - logged every day."""
    out = {f'free_gb{p.replace("/", "_")}': round(shutil.disk_usage(p).free / 1e9, 1)
           for p in ('/kaggle/working', '/tmp', '/')}
    with open('/proc/meminfo') as fh:
        mem = dict(line.split(':', 1) for line in fh)
    out['ram_avail_gb'] = round(int(mem['MemAvailable'].split()[0]) / 1e6, 1)
    return out

print('segments', wanted_segments(cfg), '| cpus', os.cpu_count(), '|', resources())

In [ ]:
t_start, timings = time.time(), []
for day in pd.date_range(START, END).date:
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print('time budget reached, stopping before', day); break
    t0 = time.time(); raw = pipeline.himawari_download(cfg, day)
    t1 = time.time(); frames = pipeline.himawari_process(cfg, day)
    t2 = time.time()
    timings.append({'day': str(day), 'files': len(raw), 'gb': raw['size_bytes'].sum() / 1e9 if len(raw) else 0,
                    'download_s': round(t1 - t0), 'process_s': round(t2 - t1),
                    'frames_ok': int((frames['status'] == 'ok').sum()) if len(frames) else 0,
                    'frames_total': len(frames)} | resources())
    logging.info('DAY DONE %s', timings[-1])
    pd.DataFrame(timings).to_csv('/kaggle/working/timings.csv', index=False)
    if timings[-1]['frames_total'] and not timings[-1]['frames_ok']:
        raise RuntimeError(f'no frame of {day} processed - see warnings above; stopping instead of wasting the run')

In [ ]:
print(pipeline.build_sat(cfg))
from stormcast.report import build_report
print(open(build_report(cfg)).read())
t = pd.DataFrame(timings)
print('GB downloaded:', round(t.gb.sum(), 1), '| frames ok:', t.frames_ok.sum(), '/', t.frames_total.sum(),
      '| median min/day:', round((t.download_s + t.process_s).median() / 60, 1))
shutil.rmtree(cfg.paths.cache, ignore_errors=True)  # resampling caches are not worth keeping as output
!du -sh /kaggle/working/data/*